<a href="https://colab.research.google.com/github/diaoumardia2001-beep/DI-Bootcamp-May/blob/main/Daily_Challenge_MCP_Airbnb_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Notebook - MCP + Airbnb (Colab)

Reference notebook: local notes MCP + Airbnb MCP + optional real LLM.

## Install
Run once. npm only needed for the real Airbnb server.

In [19]:
!pip install --force-reinstall mcp nest_asyncio requests
!pip install azure-ai-inference

# Optional: real Airbnb server
!npm install -g @openbnb/mcp-server-airbnb

  Using cached mcp-1.28.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pydantic_settings-2.14.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pyjwt-2.13.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached sse_starlette-3.4.5-py3-none-any.whl.metadata (15 kB)
  Using cached starlette-1.3.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸npm warn deprecated whatwg-encoding@3.1.1: Use @exodus/bytes instead for a more spec-conformant and faster implementation
⠸⠼⠴npm warn deprecated node-domexception@1.0.0: Use your platform's native DOMException instead
⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
changed 124 packages in 7s
⠇
⠇51 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [20]:
try:
    import mcp
    print("✅ Module 'mcp' successfully imported.")
except ImportError:
    print("❌ Module 'mcp' not found. Please re-run the `!pip install mcp` cell (cell `35a5a9b2`) and ensure it completes without errors.")

✅ Module 'mcp' successfully imported.


In [21]:
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    # stdout → 1, stderr → 2
    if self is sys.stderr:
        return 2
    return 1

# Patch the class for all OutStream instances
OutStream.fileno = _patched_fileno

# And patch the current instances explicitly
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2


## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [22]:

import os
from pathlib import Path

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_AIRBNB = True  # True if npm server available
USE_REAL_LLM = True     # True if GITHUB_TOKEN set


In [23]:
import os
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN


In [24]:
import os

# 1. Tentative d'importation sécurisée de l'API des secrets de Colab
try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False
    print("⚠️ Vous n'êtes pas dans un environnement Google Colab. Les secrets 'userdata' ne sont pas disponibles.")

# 2. Récupération et configuration du token
if colab_env:
    try:
        # Récupère le secret depuis l'onglet "Secrets" (icône clé 🔑) de Colab
        github_token = userdata.get("GITHUB_TOKEN")

        if github_token:
            # Assigne le token à la variable d'environnement système
            os.environ["GITHUB_TOKEN"] = github_token
            print("✅ GITHUB_TOKEN récupéré avec succès depuis les secrets Colab.")
        else:
            print("⚠️ Le secret 'GITHUB_TOKEN' existe mais sa valeur est vide.")

    except userdata.SecretNotFoundError:
        print("❌ Erreur : Le secret 'GITHUB_TOKEN' n'a pas été trouvé dans vos secrets Google Colab.")
        print("👉 Étape pour corriger :")
        print("   1. Cliquez sur l'icône de clé (🔑 Secrets) dans la barre latérale gauche de Colab.")
        print("   2. Ajoutez un secret avec le nom exact : GITHUB_TOKEN")
        print("   3. Collez votre token de connexion GitHub dans la valeur.")
        print("   4. Activez la case 'Accès au notebook' (Notebook access) pour ce secret.")
    except Exception as e:
        print(f"❌ Une erreur inattendue est survenue : {e}")

# 3. Vérification finale pour Python
token_is_visible = bool(os.getenv("GITHUB_TOKEN"))
print(f"\n[VÉRIFICATION] GITHUB_TOKEN visible par Python : {token_is_visible}")

❌ Erreur : Le secret 'GITHUB_TOKEN' n'a pas été trouvé dans vos secrets Google Colab.
👉 Étape pour corriger :
   1. Cliquez sur l'icône de clé (🔑 Secrets) dans la barre latérale gauche de Colab.
   2. Ajoutez un secret avec le nom exact : GITHUB_TOKEN
   3. Collez votre token de connexion GitHub dans la valeur.
   4. Activez la case 'Accès au notebook' (Notebook access) pour ce secret.

[VÉRIFICATION] GITHUB_TOKEN visible par Python : False


## Local notes MCP server

In [25]:
LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP
import uvicorn # Import uvicorn for direct server control

notes = []
mcp = FastMCP()

@mcp.tool()
def add_note(text: str) -> str:
    'Add a note to the in-memory list.'
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool()
def list_notes() -> str:
    'List saved notes.'
    if not notes:
        return "No notes yet"
    return "".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    # Use uvicorn directly to ensure the server blocks and runs indefinitely
    # mcp.app holds the FastAPI application instance
    uvicorn.run(mcp.app, host="127.0.0.1", port=8000, log_level="info")
'''.strip() + "",
    encoding="utf-8",
)
print("wrote", LOCAL_SERVER)

wrote local_notes_server.py


## Client helpers (convert tools, stub planner, optional real LLM)

In [26]:
import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def convert_tool(tool, prefix: str):
    # Azure requires ^[a-zA-Z0-9_\.-]+$, so no slashes
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")

    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))

    # Résolution du TO-DO : Appel complet de l'API Azure AI Inference
    resp = client.complete(
        messages=[
            {"role": "system", "content": "You are a helpful assistant with access to tools."},
            {"role": "user", "content": prompt}
        ],
        tools=functions if functions else None,
        tool_choice="auto" if functions else None,
        model="gpt-4o"  # Spécifiez le modèle cible (ex: gpt-4o, gpt-4o-mini, Llama-3...)
    )

    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [27]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    import os
    import json

    # MINIMAL FIX: shrink tool_results before sending to gpt-4o
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )


    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        # use the shrunk version here
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)


## Orchestrate (connect both servers and execute tool_calls)

In [28]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import sys # Import the sys module to use sys.stderr

async def orchestrate(prompt: str):
    local_params = StdioServerParameters(
        command="mcp",
        args=["run", str(LOCAL_SERVER)],
        env=BASE_ENV,  # <- use merged env
    )

    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=BASE_ENV,
        )


    async with stdio_client(local_params, errlog=sys.stderr) as (lr, lw): # Added errlog
        async with ClientSession(lr, lw) as local_sess:
            await local_sess.initialize()
            local_tools = await local_sess.list_tools()

            async with stdio_client(airbnb_params, errlog=sys.stderr) as (ar, aw): # Added errlog
                async with ClientSession(ar, aw) as airbnb_sess:
                    await airbnb_sess.initialize()
                    airbnb_tools = await airbnb_sess.list_tools()

                    functions = (
                        [convert_tool(t, "notes") for t in local_tools.tools]
                        + [convert_tool(t, "airbnb") for t in airbnb_tools.tools]
                    )

                    tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
                    print("tool_calls:", tool_calls)

                    tool_results = []
                    for call in tool_calls:
                        name = call["name"]
                        args = call["args"]
                        prefix, tool_name = name.split("__", 1)

                        if prefix == "notes":
                            res = await local_sess.call_tool(tool_name, args)
                            tool_results.append(
                                {
                                    "name": name,
                                    "args": args,
                                    "content": [c.text for c in res.content if hasattr(c, "text")],
                                }
                            )
                        elif prefix == "airbnb":
                            res = await airbnb_sess.call_tool(tool_name, args)
                            payload = []
                            if hasattr(res, "content"):
                                for c in res.content:
                                    if hasattr(c, "text"):
                                        payload.append(c.text)
                            tool_results.append(
                                {
                                    "name": name,
                                    "args": args,
                                    "content": payload,
                                }
                            )

                    # note: DO NOT call answer_with_llm here if you want to inspect things first
                    return tool_calls, tool_results

## Demo
Adjust the prompt as you like. Switch `USE_REAL_AIRBNB/USE_REAL_LLM` to true when ready.

In [ ]:
prompt = "What tools can you access ? list them please "

tool_calls, tool_results = await orchestrate(prompt)

print("tool_calls:", tool_calls)
print("tool_results:", tool_results)

In [30]:
import subprocess
import os

print("Attempting to run local_notes_server.py directly:")

try:
    # Use Popen to capture stdout and stderr separately
    process = subprocess.Popen(
        ["mcp", "run", str(LOCAL_SERVER)],
        env=BASE_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True # Decode stdout/stderr as text
    )

    # Try to read output quickly without blocking indefinitely
    # Give it a short time to start and produce output
    stdout_output, stderr_output = process.communicate(timeout=5)

    print("--- STDOUT from local_notes_server ---")
    print(stdout_output)
    print("--- STDERR from local_notes_server ---")
    print(stderr_output)

    if process.returncode is not None:
        print(f"Process exited with code: {process.returncode}")
        if process.returncode != 0:
            print("❌ Local notes server failed to start correctly.")
    else:
        print("✅ Local notes server process started (may be running in background).")
        print("You might need to manually stop it later if it's still running.")

except FileNotFoundError:
    print("❌ Error: 'mcp' command not found. Ensure MCP is installed and in your PATH.")
except subprocess.TimeoutExpired:
    print("✅ Local notes server started successfully within timeout (running in background).")
    print("To stop it, you might need to use `process.terminate()` or kill the process.")
    # If it times out, it means it's likely running in the background as intended for a server
    # We should keep a reference to 'process' to terminate it later if needed
    # For this diagnostic, we'll just report it's running.
    process.terminate()
    print("Terminating the diagnostic process.")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")



Attempting to run local_notes_server.py directly:
--- STDOUT from local_notes_server ---

--- STDERR from local_notes_server ---

Process exited with code: 0
